# ⚔️ SmolLM2-1.7B Alignment Arena — Live Side-by-Side Demo

This notebook loads the **Base**, **SFT**, and **DPO** models and launches a live interactive **Gradio Arena** with real-time side-by-side responses!

### Models Compared:
1. 🔴 **Base Model (`HuggingFaceTB/SmolLM2-1.7B`):** Raw pre-trained foundation model.
2. 🟡 **SFT Model (`manojpaul9986/smollm2-1.7b-sft-lora`):** Supervised fine-tuned on SmolTalk.
3. 🟢 **DPO Model (`manojpaul9986/smollm2-1.7b-dpo-lora`):** Preference-aligned on UltraFeedback with step-by-step reasoning.

### Step 1: Install Required Dependencies

In [ ]:
# Uninstall conflicting audio/vision packages that trigger Kaggle CUDA mismatch crashes
!pip uninstall -y torchao torchvision torchaudio
!pip install -q -U transformers trl datasets accelerate peft evaluate bitsandbytes huggingface_hub gradio pandas tabulate


### Step 2: Download SFT & DPO Adapters from Hugging Face Hub

In [ ]:
import os
from huggingface_hub import snapshot_download

BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./work"
SFT_DIR = os.path.join(BASE_DIR, "checkpoints/sft")
DPO_DIR = os.path.join(BASE_DIR, "checkpoints/dpo")

if not os.path.exists(SFT_DIR) or not os.listdir(SFT_DIR):
    print("Downloading SFT adapter from Hugging Face Hub...")
    snapshot_download(repo_id="manojpaul9986/smollm2-1.7b-sft-lora", local_dir=SFT_DIR)
else:
    print(f"Found existing SFT adapter at: {SFT_DIR}")

if not os.path.exists(DPO_DIR) or not os.listdir(DPO_DIR):
    print("Downloading DPO adapter from Hugging Face Hub...")
    snapshot_download(repo_id="manojpaul9986/smollm2-1.7b-dpo-lora", local_dir=DPO_DIR)
else:
    print(f"Found existing DPO adapter at: {DPO_DIR}")

print("✅ All adapters ready on disk!")


### Step 3: Load Models into GPU Memory

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# 1. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n'}}"
        "{% endfor %}"
        "{% if add_generation_prompt %}{{'<|im_start|>assistant\n'}}{% endif %}"
    )

# 2. Load Base Model
print("Loading Base Model (SmolLM2-1.7B)...")
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE)
base_model.eval()

# 3. Load SFT Model
print("Loading SFT Model...")
sft_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE)
sft_model = PeftModel.from_pretrained(sft_base, SFT_DIR).to(DEVICE)
sft_model.eval()

# 4. Load DPO Model (Mounted on top of merged SFT base)
print("Loading DPO Model...")
dpo_base_raw = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE)
dpo_sft_merged = PeftModel.from_pretrained(dpo_base_raw, SFT_DIR).to(DEVICE).merge_and_unload()
dpo_model = PeftModel.from_pretrained(dpo_sft_merged, DPO_DIR).to(DEVICE)
dpo_model.eval()

print("🎉 All 3 models loaded successfully into GPU memory!")


### Step 4: Launch the Live Side-by-Side Gradio Arena

In [ ]:
import gradio as gr

def generate_single(model, prompt, chat, max_tokens, temperature, top_p):
    if chat:
        messages = [{"role": "user", "content": prompt}]
        raw_inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    else:
        raw_inputs = tokenizer(prompt, return_tensors="pt")
        
    # Safely extract 2D tensor of input_ids
    if isinstance(raw_inputs, dict) or hasattr(raw_inputs, "input_ids"):
        input_ids = raw_inputs["input_ids"] if "input_ids" in raw_inputs else raw_inputs.input_ids
    else:
        input_ids = raw_inputs
        
    if not isinstance(input_ids, torch.Tensor):
        input_ids = torch.tensor(input_ids)
        
    if input_ids.ndim == 1:
        input_ids = input_ids.unsqueeze(0)
        
    input_ids = input_ids.to(DEVICE)
        
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            max_new_tokens=int(max_tokens),
            do_sample=temperature > 0.0,
            temperature=max(float(temperature), 0.01) if temperature > 0.0 else 1.0,
            top_p=float(top_p) if temperature > 0.0 else 1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.encode("<|im_end|>")[0] if "<|im_end|>" in tokenizer.get_vocab() else tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

def arena_compare(prompt, max_tokens, temperature, top_p):
    base_res = generate_single(base_model, prompt, False, max_tokens, temperature, top_p)
    sft_res = generate_single(sft_model, prompt, True, max_tokens, temperature, top_p)
    dpo_res = generate_single(dpo_model, prompt, True, max_tokens, temperature, top_p)
    return base_res, sft_res, dpo_res

custom_css = """
.gradio-container { font-family: 'Inter', sans-serif; }
.model-box textarea { font-size: 14px; line-height: 1.6; }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), css=custom_css, title="SmolLM2 Alignment Arena") as demo:
    gr.Markdown("# ⚔️ SmolLM2-1.7B Post-Training Alignment Arena")
    gr.Markdown("Live side-by-side comparison across the full alignment lifecycle: **Base ➔ SFT ➔ DPO**.")
    
    with gr.Row():
        with gr.Column(scale=4):
            prompt_box = gr.Textbox(
                label="User Prompt",
                placeholder="Enter any reasoning, math, or instruction prompt...",
                lines=3,
                value="What is the difference between a software library and a framework? Explain with a simple analogy."
            )
            btn = gr.Button("🚀 Run Live Comparison", variant="primary", size="lg")
        with gr.Column(scale=2):
            max_tok_slider = gr.Slider(minimum=32, maximum=512, value=250, step=10, label="Max New Tokens")
            temp_slider = gr.Slider(minimum=0.0, maximum=1.0, value=0.2, step=0.05, label="Temperature")
            top_p_slider = gr.Slider(minimum=0.1, maximum=1.0, value=0.9, step=0.05, label="Top-p")
            
    with gr.Row():
        with gr.Column():
            base_out = gr.Textbox(label="🔴 1. Base Model (SmolLM2-1.7B Raw)", lines=12, elem_classes="model-box")
        with gr.Column():
            sft_out = gr.Textbox(label="🟡 2. SFT Model (SmolTalk Aligned)", lines=12, elem_classes="model-box")
        with gr.Column():
            dpo_out = gr.Textbox(label="🟢 3. DPO Model (UltraFeedback Preferred)", lines=12, elem_classes="model-box")
            
    gr.Examples(
        examples=[
            ["What is the difference between a software library and a framework? Explain with a simple analogy.", 250, 0.2, 0.9],
            ["A store has 120 apples. They sell 45 in the morning and 30 in the afternoon. How many are left?", 200, 0.0, 1.0],
            ["If I have $50 and spend 40% of it, how much do I have left? Show calculation.", 200, 0.0, 1.0],
            ["Write a polite email to a customer explaining that their shipment will be delayed by 2 days due to weather conditions.", 250, 0.3, 0.9],
            ["Explain in simple terms why the sky is blue.", 250, 0.2, 0.9],
            ["What are three practical tips for staying focused while studying?", 250, 0.3, 0.9],
        ],
        inputs=[prompt_box, max_tok_slider, temp_slider, top_p_slider]
    )
    
    btn.click(
        fn=arena_compare,
        inputs=[prompt_box, max_tok_slider, temp_slider, top_p_slider],
        outputs=[base_out, sft_out, dpo_out]
    )

demo.launch(share=True, inline=True)


### Step 5: Automated Multi-Prompt & Multi-Hyperparameter Documentation

Run this cell to sweep through prompts with varying `max_tokens`, `temperature`, and `top_p` settings. It generates outputs across **Base**, **SFT**, and **DPO**, and saves the complete dataset to `arena_showcase.csv` and `arena_showcase.md` for portfolio documentation!

In [ ]:
import pandas as pd
import json
from IPython.display import display, HTML, FileLink

# 1. Define prompts across reasoning, math, instructions, and creativity
test_suite = [
    {
        "category": "Mathematical Reasoning",
        "prompt": "If I have $50 and spend 40% of it, how much do I have left? Show calculation.",
        "max_tokens": 200,
        "temperature": 0.0,
        "top_p": 1.0
    },
    {
        "category": "Multi-Step Arithmetic",
        "prompt": "A store has 120 apples. They sell 45 in the morning and 30 in the afternoon. How many are left?",
        "max_tokens": 200,
        "temperature": 0.0,
        "top_p": 1.0
    },
    {
        "category": "Conceptual Analogy",
        "prompt": "What is the difference between a software library and a framework? Explain with a simple analogy.",
        "max_tokens": 280,
        "temperature": 0.2,
        "top_p": 0.9
    },
    {
        "category": "Scientific Explanation",
        "prompt": "Explain in simple terms why the sky is blue.",
        "max_tokens": 250,
        "temperature": 0.2,
        "top_p": 0.9
    },
    {
        "category": "Multi-Constraint Advice",
        "prompt": "What are three practical tips for staying focused while studying?",
        "max_tokens": 280,
        "temperature": 0.3,
        "top_p": 0.9
    },
    {
        "category": "Professional Communication",
        "prompt": "Write a polite email to a customer explaining that their shipment will be delayed by 2 days due to weather conditions.",
        "max_tokens": 260,
        "temperature": 0.3,
        "top_p": 0.9
    },
    {
        "category": "Creative Coding Scenario (Higher Temp Sweep)",
        "prompt": "Write a Python function to check if a string is a palindrome. Include docstrings and an example.",
        "max_tokens": 280,
        "temperature": 0.4,
        "top_p": 0.95
    }
]

results = []
print(f"🚀 Generating automated documentation for {len(test_suite)} test cases...\n")

for i, item in enumerate(test_suite, start=1):
    p = item["prompt"]
    cat = item["category"]
    max_tok = item["max_tokens"]
    temp = item["temperature"]
    tp = item["top_p"]
    
    print(f"[{i}/{len(test_suite)}] Evaluating Category: {cat} (temp={temp}, top_p={tp}, max_tokens={max_tok})")
    
    base_r = generate_single(base_model, p, False, max_tok, temp, tp)
    sft_r = generate_single(sft_model, p, True, max_tok, temp, tp)
    dpo_r = generate_single(dpo_model, p, True, max_tok, temp, tp)
    
    results.append({
        "Category": cat,
        "Prompt": p,
        "Max_Tokens": max_tok,
        "Temperature": temp,
        "Top_P": tp,
        "Base_Response": base_r,
        "SFT_Response": sft_r,
        "DPO_Response": dpo_r
    })

# 2. Convert to DataFrame and save to CSV / JSON / Markdown
df = pd.DataFrame(results)
csv_path = "arena_showcase.csv"
json_path = "arena_showcase.json"
md_path = "arena_showcase.md"

df.to_csv(csv_path, index=False)
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

# Write formatted Markdown Report
with open(md_path, "w", encoding="utf-8") as f:
    f.write("# ⚔️ SmolLM2 Alignment Arena — Qualitative Generation Showcase\n\n")
    f.write("Comprehensive side-by-side comparison across Base, SFT, and DPO under varying hyperparameters.\n\n")
    for r in results:
        f.write(f"## Category: {r['Category']}\n")
        f.write(f"**Prompt:** `{r['Prompt']}`  \n")
        f.write(f"**Hyperparameters:** `temperature={r['Temperature']}`, `top_p={r['Top_P']}`, `max_tokens={r['Max_Tokens']}`\n\n")
        f.write(f"### 🔴 Base Model\n```\n{r['Base_Response']}\n```\n\n")
        f.write(f"### 🟡 SFT Model\n```\n{r['SFT_Response']}\n```\n\n")
        f.write(f"### 🟢 DPO Model\n```\n{r['DPO_Response']}\n```\n\n")
        f.write("---\n\n")

print(f"\n✅ Successfully saved results to:")
print(f"  - {csv_path}")
print(f"  - {json_path}")
print(f"  - {md_path}")

# 3. Display interactive preview
display(HTML("<h3>📊 Generated Showcase Preview (First 2 Rows)</h3>"))
display(df[["Category", "Prompt", "Temperature", "Max_Tokens"]].head(2))
display(FileLink(csv_path))
display(FileLink(md_path))
